<a href="https://colab.research.google.com/github/Damainx22/RGV-Business-Survival/blob/main/notebooks/03_cbp_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# Notebook 03: CBP/ZBP Business Patterns Cleaning
# Purpose: Load ZIP Code Business Patterns data
#          for 2018-2022, filter to RGV zip codes,
#          calculate business density per zip code,
#          and save to data/cleaned/
# ============================================

# Mount Google Drive so we can access raw data files
from google.colab import drive
drive.mount('/content/drive')

# Standard imports
import os
import pandas as pd

# Define folder paths — same as every notebook
RAW = '/content/drive/MyDrive/rgv_business_survival/data/raw'
CLEANED = '/content/drive/MyDrive/rgv_business_survival/data/cleaned'
MERGED = '/content/drive/MyDrive/rgv_business_survival/data/merged'

print("Setup complete")
print("Raw files:", sorted(os.listdir(RAW)))

Mounted at /content/drive
Setup complete
Raw files: ['7a_504_foia_data_dictionary.xlsx', 'bds2023_sec.csv', 'bds2023_st_cty.csv', 'foia-7a-fy2010-fy2019-as-of-251231.csv', 'foia-7a-fy2020-present-as-of-251231.csv', 'zbp18detail.txt', 'zbp18detail.zip', 'zbp19detail.txt', 'zbp19detail.zip', 'zbp20detail.txt', 'zbp20detail.zip', 'zbp21detail.txt', 'zbp21detail.zip', 'zbp22detail.txt', 'zbp22detail.zip']


In [18]:
# Load the 2018 ZIP Code Business Patterns file
# encoding='latin-1' handles special characters in business names
# low_memory=False prevents pandas from guessing wrong data types on large files
df = pd.read_csv(f'{RAW}/zbp18detail.txt', encoding='latin-1', low_memory=False)

# Quick check — how many rows and what columns do we have
print(df.shape)
print(df.columns.tolist())
print(df.head(3))

(2874446, 12)
['zip', 'naics', 'est', 'n<5', 'n5_9', 'n10_19', 'n20_49', 'n50_99', 'n100_249', 'n250_499', 'n500_999', 'n1000']
    zip   naics  est  n<5 n5_9 n10_19 n20_49 n50_99 n100_249 n250_499  \
0  1001  ------  473  230   80     67     57     24       13        N   
1  1001  23----   55   39    4      3      7      N        N        N   
2  1001  236///    7    5    N      N      N      N        N        N   

  n500_999 n1000  
0        N     N  
1        N     N  
2        N     N  


In [19]:
# Convert zip column to string first — it was read as integers which drops leading zeros
# zfill(5) pads with leading zeros to ensure all zips are exactly 5 digits
# Example: 1001 becomes 01001, 78501 stays 78501
df['zip'] = df['zip'].astype(str).str.zfill(5)

# Filter to Texas zip codes only
# All Texas zip codes start with 75, 76, 77, 78, or 79
df = df[df['zip'].str.startswith(('75', '76', '77', '78', '79'))]

# Verify how many unique Texas zip codes we have
print(f'Unique Texas zip codes: {df["zip"].nunique()}')
print(f'Total rows: {len(df)}')

Unique Texas zip codes: 2190
Total rows: 214462


In [20]:
# Filter to only the '------' rows which represent total establishments across all industries
# Each zip code has one '------' row that sums up all businesses in that area
df = df[df['naics'] == '------']

# Keep only the columns we need — zip code and total establishment count
# est = total number of businesses in that zip code (our business density feature)
df = df[['zip', 'est']]

# Tag each row with the year so we can track trends across 2018-2022
df['year'] = 2018

# Verify the result — should have one row per Texas zip code
print(f'Rows: {len(df)}')
print(f'Unique zip codes: {df["zip"].nunique()}')
print(df.head())

Rows: 2190
Unique zip codes: 2190
           zip   est  year
2020050  75001  1659  2018
2020529  75002  1081  2018
2020898  75006  2167  2018
2021527  75007  1188  2018
2021921  75009   248  2018


In [21]:
# create function to do this for all years from 2018-2022
years = ['18','19','20','21','22']
all_years = []
for year in years:
  df = pd.read_csv(f'{RAW}/zbp{year}detail.txt', encoding='latin-1', low_memory=False)

  df['zip'] = df['zip'].astype(str).str.zfill(5)

  # Filter to Texas zip codes only
  # All Texas zip codes start with 75, 76, 77, 78, or 79
  df = df[df['zip'].str.startswith(('75', '76', '77', '78', '79'))]

  # Filter to only the '------' rows which represent total establishments across all industries
  # Each zip code has one '------' row that sums up all businesses in that area
  df = df[df['naics'] == '------']

  # Keep only the columns we need — zip code and total establishment count
  # est = total number of businesses in that zip code (our business density feature)
  df = df[['zip', 'est']]

  # Tag each row with the year so we can track trends across 2018-2022
  df['year'] = int('20' + year)

  df = all_years.append(df)


  Loaded 2018: 2190 zip codes
  Loaded 2019: 2196 zip codes
  Loaded 2020: 2193 zip codes
  Loaded 2021: 2184 zip codes
  Loaded 2022: 2195 zip codes


In [25]:
#combine the datasets
combined = pd.concat(all_years, ignore_index=True)

# Save to CSV
combined.to_csv(f'{CLEANED}/cbp_texas.csv', index=False)

# Print Confirmation
print(f'Saved: {len(combined)} rows')

Saved: 10958 rows
